In [1]:
import arxiv
from langchain_core .documents import Document

def load_documents(query: str, max_docs: int) -> list:
  # Construct the default API client.
  client = arxiv.Client()

  # Search for the 10 most recent articles matching the keyword "quantum."
  search = arxiv.Search(
    query = query,
    max_results = max_docs,
    # sort_by = arxiv.SortCriterion.SubmittedDate
  )

  results = client.results(search)

  docs = []
  for res in results:
    doc = Document(
        page_content = res.summary,
        metadata={
            "title": res.title,
            "source": res.entry_id,
            "published": str(res.published) if res.published else None
        }
    )

    docs.append(doc)

  return docs

In [2]:
docs = load_documents("large language models", max_docs=300)
print(f"Loaded {len(docs)} docs")
print(docs[0].page_content[:300])
print(docs[0].metadata)

Loaded 300 docs
This paper explores the advancements in making large language models (LLMs) more human-like. We focus on techniques that enhance natural language understanding, conversational coherence, and emotional intelligence in AI systems. The study evaluates various approaches, including fine-tuning with dive
{'title': 'Enhancing Human-Like Responses in Large Language Models', 'source': 'http://arxiv.org/abs/2501.05032v2', 'published': '2025-01-09 07:44:06+00:00'}


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_documents(documents):
  splitter = RecursiveCharacterTextSplitter(
      chunk_size=250,
      chunk_overlap=25
  )

  texts = splitter.split_documents(documents)
  return texts

In [4]:
chunks = chunk_documents(docs)

print(f"Documents: {len(docs)}")
print(f"Chunks: {len(chunks)}")
print(f"Avg chunks per doc: {len(chunks)/len(docs):.1f}")
print(f"\nSample chunk:\n{chunks[0].page_content}")

Documents: 300
Chunks: 1712
Avg chunks per doc: 5.7

Sample chunk:
This paper explores the advancements in making large language models (LLMs) more human-like. We focus on techniques that enhance natural language understanding, conversational coherence, and emotional intelligence in AI systems. The study evaluates


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

similar_1 = "transformer attention mechanism in neural networks"
similar_2 = "self-attention and multi-head attention in deep learning"
unrelated  = "the best way to cook pasta"

vec1 = embeddings.embed_query(similar_1)
vec2 = embeddings.embed_query(similar_2)
vec3 = embeddings.embed_query(unrelated)

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"Similar sentences:  {cosine_sim(vec1, vec2):.4f}")
print(f"Unrelated sentence: {cosine_sim(vec1, vec3):.4f}")
print(f"Vector dimensions:  {len(vec1)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Similar sentences:  0.5234
Unrelated sentence: 0.0965
Vector dimensions:  384


In [8]:
import os
from langchain_chroma import Chroma
import hashlib

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

def get_vectorstore():
  vs = Chroma(
      collection_name="foo",
      embedding_function=embeddings,
      persist_directory="./data/chroma_db"
  )

  return vs


def add_documents(chunks):
    vs = get_vectorstore()

    # Create hash IDs from document content
    ids = [
        hashlib.md5(
            doc.page_content.encode("utf-8")
        ).hexdigest()
        for doc in chunks
    ]

    vs.add_documents(
        documents=chunks,
        ids=ids
    )

    return vs

vs = add_documents(chunks)
print(f"Added {len(chunks)} documents")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Added 1712 documents


In [11]:
import time
def test_retrieval(vectorstore, query, k):
  start = time.time()
  query_res = vectorstore.similarity_search_with_score(
      query = query,
      k = k
  )
  end = time.time()
  print("Execution time for query - " + query + ":", end - start, "seconds")

  return query_res

test_retrieval(vs, "transformer attention mechanism", 10)

Execution time for query - transformer attention mechanism: 0.07720279693603516 seconds


[(Document(id='1bdb161ab5d3d886d36b4dabb1438208', metadata={'source': 'http://arxiv.org/abs/2402.17762v2', 'published': '2024-02-27 18:55:17+00:00', 'title': 'Massive Activations in Large Language Models'}, page_content='in LLMs. Third, these massive activations lead to the concentration of attention probabilities to their corresponding tokens, and further, implicit bias terms in the self-attention output. Last, we also study massive activations in Vision'),
  0.8555037975311279),
 (Document(id='71334047e607983589a51a39cdf6b60c', metadata={'published': '2023-02-10 18:21:13+00:00', 'source': 'http://arxiv.org/abs/2302.05406v1', 'title': 'Adversarial Transformer Language Models for Contextual Commonsense Inference'}, page_content='lack of controllability for topics of the inferred facts; lack of commonsense knowledge during training; and, possibly, hallucinated or false facts. In this work, we utilize a transformer model for this task and develop techniques to address the'),
  0.96023929